# BrainInsight AI
# Notebook 04: Model Training

## Objectives
Train multiple classical machine learning models using the extracted handcrafted features.

### Models
- Decision Tree
- K-Nearest Neighbors (KNN)
- XGBoost
- Random Forest

The trained models will be saved in the `saved_models/` directory.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)

X = np.load(OUTPUT_DIR / "X.npy")
y = np.load(OUTPUT_DIR / "y.npy", allow_pickle=True)

print("Feature Matrix:", X.shape)
print("Labels:", y.shape)


Feature Matrix: (7200, 34658)
Labels: (7200,)


## Encode Labels

In [2]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

joblib.dump(encoder, MODEL_DIR / "label_encoder.pkl")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])


Training Samples : 5760
Testing Samples  : 1440


## Define Models

In [3]:
models = {

    "decision_tree": DecisionTreeClassifier(
        criterion="gini",
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ),

    "knn": KNeighborsClassifier(
        n_neighbors=3,
        weights="distance"
    ),

    "xgboost": XGBClassifier(
        n_estimators=75,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )
}

## Train Models

In [4]:
trained_models = {}

for name, model in models.items():

    print("=" * 60)
    print(f"Training: {name}")

    model.fit(X_train, y_train)

    trained_models[name] = model

    joblib.dump(model, MODEL_DIR / f"{name}.pkl")

    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)

    print(f"Training Accuracy : {train_acc:.4f}")
    print(f"Testing Accuracy  : {test_acc:.4f}")


Training: decision_tree
Training Accuracy : 0.9736
Testing Accuracy  : 0.7000
Training: knn
Training Accuracy : 1.0000
Testing Accuracy  : 0.6903
Training: xgboost
Training Accuracy : 0.9608
Testing Accuracy  : 0.8146
Training: random_forest
Training Accuracy : 1.0000
Testing Accuracy  : 0.8201


## Saved Models

In [5]:
print("Saved Models:")

for model_file in sorted(MODEL_DIR.glob("*.pkl")):
    print("-", model_file.name)


Saved Models:
- decision_tree.pkl
- knn.pkl
- label_encoder.pkl
- random_forest.pkl
- scaler.pkl
- xgboost.pkl


## Conclusion

All selected machine learning models have been successfully trained and stored. The next notebook will evaluate and compare these models using performance metrics such as Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC Curve to identify the best-performing classifier.


## Interview Notes

- Why is Label Encoding required for classification?
- Why split data into training and testing sets?
- Why use Random Forest as the expected final model?
- What are the advantages of Extra Trees over Decision Trees?
- Why compare multiple models instead of selecting one immediately?
